# CSE440 Project — Notebook 01
## Dataset, EDA, Preprocessing, Splits, and Text Representations

**Owner:** Mahir
**Covers report sections:** 3.2, 3.3, 3.4, 3.5, 3.6
**Runtime:** Google Colab (CPU is fine — no models are trained here)

This notebook is the foundation for the whole project. It produces the cached
split files and representation artifacts that every other notebook loads, so
that all ten models are trained and evaluated on exactly the same rows.

**Nothing downstream may start until Task 7 has run and the team has verified
matching checksums.**

| Task | What it does | Report section |
|---|---|---|
| 1 | Environment setup, config constants | — |
| 2 | Load dataset, describe source and structure | 3.2 |
| 3 | Statistics, data-quality issues, visualisations | 3.3 |
| 4 | Preprocessing — two strategies, compared | 3.4 |
| 5 | Train / validation / test split | 3.5 |
| 6 | Text representations — TF-IDF, Word2Vec, GloVe | 3.6 |
| 7 | Export artifacts + team verification cell | — |

---
# Task 1 — Environment setup

Installs, imports, and the frozen configuration constants. Every constant here
is shared across the four team notebooks; changing one invalidates the
comparison table.

In [ ]:
# Colab install. `datasets` and `gensim` are not preinstalled.
!pip install -q datasets gensim wordcloud

In [ ]:
import os, re, json, time, hashlib, string
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 90)
print("imports ok")

In [ ]:
# ---- Frozen configuration. Do not change without telling the whole team. ----
SEED          = 42
VAL_SIZE      = 0.15      # carved out of TRAIN only
MIN_DOC_CHARS = 10        # documents shorter than this after cleaning are dropped
MAX_VOCAB     = 20000     # Keras tokenizer vocabulary cap
MAX_LEN       = 250       # padded sequence length
EMBEDDING_DIM = 50        # matches glove.6B.50d

DATASET_NAME  = "SetFit/20_newsgroups"

np.random.seed(SEED)

# Google Drive is where artifacts live so all four members load identical files.
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR   = Path("/content/drive/MyDrive/cse440_project")
SPLIT_DIR  = DATA_DIR / "splits"
REPR_DIR   = DATA_DIR / "representations"
FIG_DIR    = DATA_DIR / "figures"
for d in (SPLIT_DIR, REPR_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"seed          : {SEED}")
print(f"artifacts dir : {DATA_DIR}")

In [ ]:
# NLTK resources used in Task 4.
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"warn: {pkg} -> {e}")
print("nltk resources ready")

---
# Task 2 — Dataset collection and description  *(Report §3.2)*

**Source.** The 20 Newsgroups corpus, distributed on the Hugging Face Hub as
`SetFit/20_newsgroups`. It is a collection of Usenet posts from 1993–1994,
partitioned into 20 discussion groups. The Hugging Face copy ships with the
original *temporal* train/test split: training posts precede test posts in
time. That split is a property of the benchmark and **must not be reshuffled** —
doing so leaks future posts into training and inflates every score.

**Task.** Single-label multi-class classification: predict which of the 20
newsgroups a post was published in.

In [ ]:
raw = load_dataset(DATASET_NAME)
print(raw)

In [ ]:
train_raw = raw["train"].to_pandas()
test_raw  = raw["test"].to_pandas()

print(f"train rows : {len(train_raw):,}")
print(f"test rows  : {len(test_raw):,}")
print(f"total rows : {len(train_raw) + len(test_raw):,}")
print(f"columns    : {list(train_raw.columns)}")
train_raw.head(3)

In [ ]:
# Label id -> name map. Frozen and exported so every notebook decodes identically.
label_map = (train_raw[["label", "label_text"]]
             .drop_duplicates()
             .sort_values("label")
             .set_index("label")["label_text"]
             .to_dict())

CLASS_NAMES = [label_map[i] for i in sorted(label_map)]
NUM_CLASSES = len(CLASS_NAMES)

print(f"{NUM_CLASSES} classes:")
for i, n in enumerate(CLASS_NAMES):
    print(f"  {i:2d}  {n}")

### Hierarchical label structure

The 20 groups are not independent — they cluster into topical families
(`comp.*`, `rec.*`, `sci.*`, `talk.*`, plus three loners). Confusions *within*
a family are qualitatively different from confusions *across* families, and the
report's error analysis depends on this distinction. The superclass map is
defined once here and reused by the shared evaluation module.

In [ ]:
SUPERCLASS = {n: (n.split(".")[0] if not n.startswith("misc") else "misc")
              for n in CLASS_NAMES}

fam = pd.Series(SUPERCLASS).value_counts().sort_index()
print("classes per family:")
print(fam.to_string())

In [ ]:
# A concrete example so the report can describe what a document actually is.
ex = train_raw.iloc[7]
print(f"label: {ex['label']}  ({ex['label_text']})")
print(f"chars: {len(ex['text'])}")
print("-" * 70)
print(ex["text"][:700])

---
# Task 3 — Data statistics and quality issues  *(Report 3.3)*

Three questions to answer with evidence: how balanced are the classes, how long
are the documents, and what is broken in the data.

In [ ]:
full = pd.concat(
    [train_raw.assign(split="train"), test_raw.assign(split="test")],
    ignore_index=True,
)
full["n_chars"] = full["text"].str.len()
full["n_words"] = full["text"].str.split().str.len()

print(full.groupby("split")[["n_chars", "n_words"]].describe().T.round(1))

### 3a. Class distribution

In [ ]:
dist = (full.groupby(["split", "label_text"]).size()
        .unstack(0).reindex(CLASS_NAMES))
dist["total"] = dist.sum(axis=1)
dist["pct"]   = (100 * dist["total"] / dist["total"].sum()).round(2)

imbalance = dist["total"].max() / dist["total"].min()
print(dist.to_string())
print(f"\nimbalance ratio (max/min) = {imbalance:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
dist[["train", "test"]].plot(kind="barh", stacked=True, ax=ax)
ax.set_xlabel("documents"); ax.set_ylabel("")
ax.set_title("Class distribution by split")
plt.tight_layout()
plt.savefig(FIG_DIR / "class_distribution.png", dpi=150)
plt.show()

The corpus is close to balanced — roughly 600–1000 documents per class. There
is no severe imbalance to correct, so no resampling or class weighting is
applied. Accuracy is therefore an honest headline metric, though macro-F1 is
still reported since the smaller religion and politics groups are the ones
models tend to confuse.

### 3b. Document length

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(full["n_words"].clip(upper=1500), bins=60)
axes[0].set_title("Document length (words, clipped at 1500)")
axes[0].set_xlabel("words"); axes[0].axvline(MAX_LEN, color="red", ls="--",
                                             label=f"MAX_LEN={MAX_LEN}")
axes[0].legend()

order = full.groupby("label_text")["n_words"].median().sort_values().index
sns.boxplot(data=full, y="label_text", x="n_words", order=order,
            showfliers=False, ax=axes[1])
axes[1].set_title("Length by class"); axes[1].set_ylabel("")
plt.tight_layout()
plt.savefig(FIG_DIR / "length_distribution.png", dpi=150)
plt.show()

cov = (full["n_words"] <= MAX_LEN).mean()
print(f"documents fully covered by MAX_LEN={MAX_LEN}: {cov:.1%}")
print(full["n_words"].quantile([.5, .75, .9, .95, .99]).round(0).to_string())

Length is heavily right-skewed: a long tail of quoted reply-chains drags the
mean far above the median. The `MAX_LEN` line shows what fraction of each
document the sequence models will actually see — record this number, because it
is the honest explanation for any ceiling the RNN tier hits.

### 3c. Data quality issues

In [ ]:
empty      = (full["text"].str.strip().str.len() == 0).sum()
near_empty = (full["text"].str.strip().str.len() < MIN_DOC_CHARS).sum()
missing    = full["text"].isna().sum()

exact_dupes = full.duplicated(subset=["text"]).sum()
cross_leak  = test_raw["text"].isin(set(train_raw["text"])).sum()

print(f"missing (NaN) texts        : {missing}")
print(f"empty texts                : {empty}")
print(f"under {MIN_DOC_CHARS} chars: {near_empty}")
print(f"exact duplicate texts      : {exact_dupes}")
print(f"test rows also in train    : {cross_leak}   <-- leakage check")

**Decisions taken.** Documents that are empty or shorter than `MIN_DOC_CHARS`
carry no signal and are removed in Task 4. Exact duplicates within *train* are
dropped so a repeated post cannot be memorised; duplicates are **not** removed
from test, because the test set must remain the untouched benchmark. Any
train/test overlap is reported honestly rather than silently deleted — it is a
property of the published benchmark, not a bug we introduced.

### 3d. Lexical view — most frequent terms per family

In [ ]:
from wordcloud import WordCloud

sw = set(stopwords.words("english"))
fams = sorted(set(SUPERCLASS.values()))
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for ax, f in zip(axes.ravel(), fams):
    members = [c for c in CLASS_NAMES if SUPERCLASS[c] == f]
    blob = " ".join(full.loc[full.label_text.isin(members), "text"]
                        .str.lower().head(400))
    wc = WordCloud(width=420, height=240, background_color="white",
                   stopwords=sw, max_words=45).generate(blob)
    ax.imshow(wc); ax.axis("off"); ax.set_title(f)

for ax in axes.ravel()[len(fams):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "wordclouds_by_family.png", dpi=150)
plt.show()

---
# Task 4 — Data preprocessing  *(Report §3.4)*

Two strategies are implemented and compared, because §3.4 asks for a
justified comparison rather than a single unexamined pipeline.

| Strategy | Steps |
|---|---|
| **light** | lowercase, strip emails / URLs / digits, strip punctuation, collapse whitespace |
| **full**  | light + stopword removal + WordNet lemmatisation |

`text_raw` is kept untouched alongside both, because BERT brings its own
subword tokeniser and must never receive text we have already stripped.

In [ ]:
EMAIL_RE = re.compile(r"\S+@\S+")
URL_RE   = re.compile(r"http\S+|www\.\S+")
NONALPHA = re.compile(r"[^a-z\s]")
WS_RE    = re.compile(r"\s+")

STOPWORDS  = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()


def clean_light(text: str) -> str:
    """Lowercase, remove emails/URLs/digits/punctuation, collapse whitespace."""
    if not isinstance(text, str):
        return ""
    t = text.lower()
    t = EMAIL_RE.sub(" ", t)
    t = URL_RE.sub(" ", t)
    t = NONALPHA.sub(" ", t)
    return WS_RE.sub(" ", t).strip()


def lemma(word: str) -> str:
    """Noun lemma first, then verb lemma.

    WordNetLemmatizer assumes a noun unless told otherwise, so a single call
    turns 'drives' into 'drive' but leaves 'driving' untouched. Chaining the
    verb pass collapses both onto 'drive', which is the whole point of
    lemmatising here.
    """
    return LEMMATIZER.lemmatize(LEMMATIZER.lemmatize(word), "v")


def clean_full(text: str) -> str:
    """clean_light, then drop stopwords and lemmatise the survivors."""
    toks = clean_light(text).split()
    toks = [lemma(w) for w in toks if w not in STOPWORDS and len(w) > 2]
    return " ".join(toks)


demo = train_raw["text"].iloc[7]
print("RAW  :", demo[:220].replace("\n", " "), "\n")
print("LIGHT:", clean_light(demo)[:220], "\n")
print("FULL :", clean_full(demo)[:220], "\n")

# Evidence that inflections actually collapse to one feature.
print("lemmatisation check:",
      {w: lemma(w) for w in ["drive", "drives", "driving", "cars", "running"]})

In [ ]:
t0 = time.time()
for df in (train_raw, test_raw):
    df["text_raw"]   = df["text"]
    df["text_light"] = df["text"].map(clean_light)
    df["text_clean"] = df["text"].map(clean_full)   # `text_clean` = the full strategy
print(f"preprocessed both splits in {time.time() - t0:.1f}s")

### Comparing the two strategies

In [ ]:
def vocab_of(series, cap=None):
    fd = FreqDist(w for doc in series for w in doc.split())
    return fd

fd_light = vocab_of(train_raw["text_light"])
fd_full  = vocab_of(train_raw["text_clean"])

comp = pd.DataFrame({
    "vocabulary size": [len(fd_light), len(fd_full)],
    "total tokens":    [fd_light.N(), fd_full.N()],
    "mean tokens/doc": [train_raw["text_light"].str.split().str.len().mean(),
                        train_raw["text_clean"].str.split().str.len().mean()],
    "hapax (freq==1)": [sum(1 for c in fd_light.values() if c == 1),
                        sum(1 for c in fd_full.values() if c == 1)],
}, index=["light", "full"]).round(1)

print(comp.to_string())
print("\ntop 15 terms — light:", [w for w, _ in fd_light.most_common(15)])
print("top 15 terms — full :", [w for w, _ in fd_full.most_common(15)])

**Justification for choosing `full` as the default.** Stopword removal strips
the high-frequency function words that dominate the light vocabulary and carry
no topical signal, and lemmatisation collapses inflections so that *drive*,
*drives* and *driving* become one feature rather than three sparse ones. The
result is a materially smaller vocabulary with fewer hapax terms — which matters
directly, since `MAX_VOCAB` is capped at 20,000 and every slot spent on a
stopword or an inflected variant is a slot not spent on a topical term.

`text_light` is retained in the exported files so the classical tier can run one
config on each strategy and report the empirical difference, rather than resting
the argument on reasoning alone.

In [ ]:
# Apply the quality decisions from Task 3c.
before_tr, before_te = len(train_raw), len(test_raw)

train_raw = train_raw[train_raw["text_clean"].str.len() >= MIN_DOC_CHARS].copy()
test_raw  = test_raw[test_raw["text_clean"].str.len() >= MIN_DOC_CHARS].copy()

train_raw = train_raw.drop_duplicates(subset=["text_clean"]).copy()  # train only

print(f"train : {before_tr:,} -> {len(train_raw):,}  "
      f"(-{before_tr - len(train_raw):,})")
print(f"test  : {before_te:,} -> {len(test_raw):,}  "
      f"(-{before_te - len(test_raw):,})")

---
# Task 5 — Train / validation / test split  *(Report §3.5)*

The published train/test boundary is **temporal** and is preserved exactly.
The validation set is carved out of training only, stratified by label so every
class keeps its proportion, with a fixed seed for reproducibility.

Tuning is guided by validation performance; the test set is touched once, at the
end, for the final numbers. Tuning against test would be indefensible at the
viva.

In [ ]:
train_df, val_df = train_test_split(
    train_raw,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_raw["label"],
)

test_df = test_raw.copy()

cols = ["text_raw", "text_light", "text_clean", "label", "label_text"]
train_df = train_df[cols].reset_index(drop=True)
val_df   = val_df[cols].reset_index(drop=True)
test_df  = test_df[cols].reset_index(drop=True)

n = len(train_df) + len(val_df) + len(test_df)
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} : {len(df):6,}  ({len(df)/n:5.1%})")

In [ ]:
# Stratification check — proportions should track across train and val.
chk = pd.DataFrame({
    "train %": train_df.label_text.value_counts(normalize=True) * 100,
    "val %":   val_df.label_text.value_counts(normalize=True) * 100,
    "test %":  test_df.label_text.value_counts(normalize=True) * 100,
}).reindex(CLASS_NAMES).round(2)
chk["train-val gap"] = (chk["train %"] - chk["val %"]).abs().round(2)
print(chk.to_string())
print(f"\nmax train/val gap: {chk['train-val gap'].max():.2f} pp")

In [ ]:
# No document may appear in more than one split.
s_tr, s_va, s_te = (set(d.text_clean) for d in (train_df, val_df, test_df))
print(f"train n val : {len(s_tr & s_va)}")
print(f"train n test: {len(s_tr & s_te)}")
print(f"val   n test: {len(s_va & s_te)}")

---
# Task 6 — Text representation  *(Report §3.6)*

Three representations are built; §3.6 requires at least two. Each is evaluated
*intrinsically* here — sparsity, vocabulary coverage, and nearest-neighbour
sanity checks. Predictive comparison happens in the modelling notebooks, where
the same splits are reused.

All three are fitted on **training data only**. Fitting a vectoriser on the full
corpus leaks test-set vocabulary statistics into training.

### 6a. TF-IDF

In [ ]:
tfidf = TfidfVectorizer(
    max_features=MAX_VOCAB,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.85,
    sublinear_tf=True,
)

X_train_tfidf = tfidf.fit_transform(train_df["text_clean"])   # fit on TRAIN only
X_val_tfidf   = tfidf.transform(val_df["text_clean"])
X_test_tfidf  = tfidf.transform(test_df["text_clean"])

density = X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])
print(f"train matrix : {X_train_tfidf.shape}")
print(f"val matrix   : {X_val_tfidf.shape}")
print(f"test matrix  : {X_test_tfidf.shape}")
print(f"density      : {density:.4%}  (sparse, as expected)")

In [ ]:
# Highest-weight terms per class — a readable check that TF-IDF found topic words.
import numpy as np
feats = np.array(tfidf.get_feature_names_out())
for c in [0, 5, 11, 17]:
    rows = X_train_tfidf[(train_df.label == c).values]
    top = feats[np.asarray(rows.mean(axis=0)).ravel().argsort()[::-1][:10]]
    print(f"{CLASS_NAMES[c]:26s} {', '.join(top)}")

### 6b. Keras tokenizer — the index both embedding matrices align to

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text_clean"])          # TRAIN only

def to_padded(texts):
    return pad_sequences(tokenizer.texts_to_sequences(texts),
                         maxlen=MAX_LEN, padding="post", truncating="post")

X_train_seq = to_padded(train_df["text_clean"])
X_val_seq   = to_padded(val_df["text_clean"])
X_test_seq  = to_padded(test_df["text_clean"])

vocab_size = min(MAX_VOCAB, len(tokenizer.word_index) + 1)
print(f"words seen      : {len(tokenizer.word_index):,}")
print(f"vocab_size used : {vocab_size:,}")
print(f"sequence shapes : {X_train_seq.shape} {X_val_seq.shape} {X_test_seq.shape}")

### 6c. Word2Vec — embeddings learned on this corpus

In [ ]:
from gensim.models import Word2Vec

sentences = [t.split() for t in train_df["text_clean"]]

w2v = Word2Vec(
    sentences,
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=2,
    workers=4,
    seed=SEED,
    epochs=10,
)
print(f"w2v vocabulary : {len(w2v.wv):,} words")

for probe in ["god", "car", "encryption", "space"]:
    if probe in w2v.wv:
        near = [w for w, _ in w2v.wv.most_similar(probe, topn=6)]
        print(f"{probe:12s} -> {', '.join(near)}")

### 6d. GloVe — pretrained general-domain embeddings

In [ ]:
GLOVE_PATH = DATA_DIR / "glove.6B.50d.txt"

if not GLOVE_PATH.exists():
    print("downloading GloVe (~820MB zip, one time — then it lives in Drive)")
    !wget -q -nc http://nlp.stanford.edu/data/glove.6B.zip -O /content/glove.6B.zip
    !unzip -o -q /content/glove.6B.zip glove.6B.50d.txt -d /content/
    !cp /content/glove.6B.50d.txt "{GLOVE_PATH}"

glove = {}
with open(GLOVE_PATH, encoding="utf8") as f:
    for line in f:
        parts = line.rstrip().split(" ")
        glove[parts[0]] = np.asarray(parts[1:], dtype="float32")
print(f"glove vectors loaded: {len(glove):,}")

In [ ]:
def build_matrix(lookup, name):
    """Align vectors to the Keras tokenizer index. Row 0 stays zero (padding)."""
    M = np.zeros((vocab_size, EMBEDDING_DIM), dtype="float32")
    hits = 0
    for word, idx in tokenizer.word_index.items():
        if idx >= vocab_size:
            continue
        vec = lookup(word)
        if vec is not None:
            M[idx] = vec
            hits += 1
    cov = hits / (vocab_size - 1)
    print(f"{name:9s} coverage: {hits:,}/{vocab_size - 1:,} = {cov:.1%}")
    return M, cov

emb_w2v,   cov_w2v   = build_matrix(lambda w: w2v.wv[w] if w in w2v.wv else None,
                                    "Word2Vec")
emb_glove, cov_glove = build_matrix(lambda w: glove.get(w), "GloVe")

**Read this coverage number carefully — it is the sharpest finding in §3.6.**
Word2Vec is trained on this corpus, so it has a vector for essentially every
in-vocabulary word. GloVe was trained on Wikipedia and Gigaword and has never
seen 1993 Usenet jargon, misspellings, or the fragments of quoted email that
survive cleaning; every miss becomes a zero row, which the model reads as a word
with no meaning. The gap between the two coverage figures is the mechanism
behind whatever accuracy difference the RNN tier reports later.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(["Word2Vec", "GloVe"], [cov_w2v * 100, cov_glove * 100],
            color=["#4c72b0", "#dd8452"])
axes[0].set_ylabel("% of vocabulary covered"); axes[0].set_ylim(0, 100)
axes[0].set_title("Embedding vocabulary coverage")

zero_w2v   = (emb_w2v.sum(axis=1) == 0).sum()
zero_glove = (emb_glove.sum(axis=1) == 0).sum()
axes[1].bar(["Word2Vec", "GloVe"], [zero_w2v, zero_glove],
            color=["#4c72b0", "#dd8452"])
axes[1].set_ylabel("zero rows in embedding matrix")
axes[1].set_title("Words with no vector")
plt.tight_layout()
plt.savefig(FIG_DIR / "embedding_coverage.png", dpi=150)
plt.show()

---
# Task 7 — Export artifacts and verify

Everything downstream loads these files. Once they are written, post in the team
channel and have **all four** members run the verification cell at the bottom.
Matching checksums are the green light for Phase 2.

In [ ]:
train_df.to_parquet(SPLIT_DIR / "train.parquet", index=False)
val_df.to_parquet(SPLIT_DIR / "val.parquet",   index=False)
test_df.to_parquet(SPLIT_DIR / "test.parquet",  index=False)

with open(SPLIT_DIR / "label_map.json", "w") as f:
    json.dump({"classes": CLASS_NAMES,
               "superclass": SUPERCLASS,
               "num_classes": NUM_CLASSES}, f, indent=2)

print("splits written:")
for p in sorted(SPLIT_DIR.iterdir()):
    print(f"  {p.name:20s} {p.stat().st_size/1e6:7.2f} MB")

In [ ]:
import pickle
from scipy import sparse

sparse.save_npz(REPR_DIR / "tfidf_train.npz", X_train_tfidf)
sparse.save_npz(REPR_DIR / "tfidf_val.npz",   X_val_tfidf)
sparse.save_npz(REPR_DIR / "tfidf_test.npz",  X_test_tfidf)
with open(REPR_DIR / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

np.save(REPR_DIR / "seq_train.npy", X_train_seq)
np.save(REPR_DIR / "seq_val.npy",   X_val_seq)
np.save(REPR_DIR / "seq_test.npy",  X_test_seq)
with open(REPR_DIR / "tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

np.save(REPR_DIR / "emb_word2vec.npy", emb_w2v)
np.save(REPR_DIR / "emb_glove.npy",    emb_glove)
w2v.save(str(REPR_DIR / "word2vec.model"))

with open(REPR_DIR / "repr_meta.json", "w") as f:
    json.dump({"vocab_size": int(vocab_size), "max_len": MAX_LEN,
               "embedding_dim": EMBEDDING_DIM, "max_vocab": MAX_VOCAB,
               "coverage_word2vec": float(cov_w2v),
               "coverage_glove": float(cov_glove)}, f, indent=2)

print("representations written:")
for p in sorted(REPR_DIR.iterdir()):
    print(f"  {p.name:24s} {p.stat().st_size/1e6:7.2f} MB")

### Team verification cell

Send this to Hasib, Fabiha, and Zawad. All four outputs must be **identical**.
If any line differs, stop — do not start training, because two people are
holding different data and the comparison table would be meaningless.

In [ ]:
import pandas as pd, json
from pathlib import Path

SPLIT_DIR = Path("/content/drive/MyDrive/cse440_project/splits")

tr = pd.read_parquet(SPLIT_DIR / "train.parquet")
va = pd.read_parquet(SPLIT_DIR / "val.parquet")
te = pd.read_parquet(SPLIT_DIR / "test.parquet")

print(f"shapes      : {tr.shape} {va.shape} {te.shape}")
print(f"label counts: {tr.label.value_counts().sort_index().tolist()}")
print(f"first clean : {tr.text_clean.iloc[0][:80]!r}")
print(f"checksum tr : {pd.util.hash_pandas_object(tr.text_clean).sum()}")
print(f"checksum te : {pd.util.hash_pandas_object(te.text_clean).sum()}")

---
## Where this leaves the project

Phase 1 is complete: sections 3.2–3.6 are evidenced, and Drive holds the splits,
the TF-IDF matrices, the tokenizer, the padded sequences, and both embedding
matrices.

**Next, in parallel:**
- **Mahir** — TF-IDF + RandomForest / LogisticRegression / MultinomialNB, 9 runs
- **Hasib** — SimpleRNN and Bi-SimpleRNN on both embedding matrices, 6 runs
- **Fabiha** — LSTM and Bi-LSTM, 6 runs, plus `utils/evaluation.py` for everyone
- **Zawad** — GRU and Bi-GRU, 6 runs, and BERT on `text_raw` (Colab GPU), 3 runs

Agree the three hyperparameter configurations before anyone trains, vary the
same dimension across all six sequence models, and log every run to your own
CSV the moment it finishes.